# 04 - Spectral Clustering

This notebook applies spectral clustering to the Paddy farm similarity graph. The graph was constructed from the first 6 PCA components using Gaussian similarity weights.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.clustering import run_spectral_workflow

PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

## Run Spectral Clustering

Candidate cluster counts from `k=2` through `k=10` are evaluated using internal validation metrics. The default selected value is the one with the highest silhouette score.

In [2]:
result = run_spectral_workflow(
    pca_coordinates_path=PROCESSED_DIR / "pca_coordinates_all_components.csv",
    graph_edges_path=PROCESSED_DIR / "similarity_graph_edges.csv",
    pca_2d_path=PROCESSED_DIR / "pca_coordinates_2d.csv",
    output_dir=PROCESSED_DIR,
    table_dir=TABLE_DIR,
    figure_dir=FIGURE_DIR,
    n_components=6,
    min_k=2,
    max_k=10,
    random_state=42,
)

result.metadata

{'selected_method': 'spectral',
 'selected_k': 10,
 'selection_metric': 'highest silhouette_score',
 'candidate_k_values': [2, 3, 4, 5, 6, 7, 8, 9, 10],
 'rows_clustered': 2789,
 'pca_components_used': 6,
 'selected_silhouette_score': 0.5886323149184591,
 'selected_calinski_harabasz_score': 1901.5714747938491,
 'selected_davies_bouldin_score': 0.7364829354445894,
 'selected_cluster_sizes': {'0': 313,
  '1': 368,
  '2': 218,
  '3': 323,
  '4': 237,
  '5': 344,
  '6': 313,
  '7': 374,
  '8': 97,
  '9': 202}}

## Cluster Validation Metrics

Higher silhouette and Calinski-Harabasz scores are better. Lower Davies-Bouldin scores are better.

In [3]:
result.metrics

,method,n_clusters,silhouette_score,calinski_harabasz_score,davies_bouldin_score,smallest_cluster_size,largest_cluster_size
0,spectral,2,0.305529,1015.273550,1.476565,936,1853
1,spectral,3,0.350402,1031.487693,1.449616,823,1030
2,spectral,4,0.403659,1014.052235,1.204411,410,936
3,spectral,5,0.454562,1107.641516,1.089172,410,838
4,spectral,6,0.528693,1307.810843,0.886586,410,605
5,spectral,7,0.530483,1411.983521,0.817241,276,486
6,spectral,8,0.543767,1555.371826,0.813636,218,473
7,spectral,9,0.562134,1710.306880,0.847850,197,374
8,spectral,10,0.588632,1901.571475,0.736483,97,374


## Selected Cluster Sizes

In [4]:
result.labels["selected_spectral_cluster"].value_counts().sort_index().rename("farm_count")

selected_spectral_cluster
0    313
1    368
2    218
3    323
4    237
5    344
6    313
7    374
8     97
9    202
Name: farm_count, dtype: int64

## Saved Outputs

In [5]:
for path in [
    PROCESSED_DIR / "spectral_cluster_labels.csv",
    TABLE_DIR / "spectral_clustering_metrics.csv",
    TABLE_DIR / "spectral_clustering_metadata.json",
    FIGURE_DIR / "spectral_silhouette_by_k.png",
    FIGURE_DIR / "spectral_cluster_sizes.png",
    FIGURE_DIR / "spectral_clusters_pca_2d.png",
]:
    print(path.relative_to(PROJECT_ROOT))

outputs/processed/spectral_cluster_labels.csv
outputs/tables/spectral_clustering_metrics.csv
outputs/tables/spectral_clustering_metadata.json
outputs/figures/spectral_silhouette_by_k.png
outputs/figures/spectral_cluster_sizes.png
outputs/figures/spectral_clusters_pca_2d.png
